In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class NoisyLinear(nn.Module):
    # y=(mu_w + sigma_w * epsolon_w)x + mu_b + sigma_b * epsilon_b
    def __init__(self, in_features, out_features, std_init=0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(
            torch.empty(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))

        self.register_buffer(
            'weight_epsilon', torch.empty(out_features, in_features))
        self.register_buffer('bias_epsilon', torch.empty(out_features))

        mu_range = 1 / math.sqrt(in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.weight_sigma.data.fill_(std_init / math.sqrt(in_features))
        self.bias_mu.data.uniform_(-mu_range, mu_range)
        self.bias_sigma.data.fill_(std_init / math.sqrt(out_features))

        self.reset_noise()
    
    def reset_noise(self,):
        # f(x) = sign(x) * sqrt(|x|)
        f = lambda x: x.sign().mul_(x.abs().sqrt())
        eps_in = f(torch.randn(self.in_features))
        eps_out = f(torch.randn(self.out_features))

        self.weight_epsilon.copy_(eps_out.outer(eps_in))
        self.bias_epsilon.copy_(eps_out)
        
    def forward(self, x):
        if self.training:
            weight = self.weight_mu + self.weight_sigma*self.weight_epsilon
            bias = self.bias_mu + self.bias_sigma*self.bias_epsilon
        else: 
            weight, bias = self.weight_mu, self.bias_mu
        return F.linear(x, weight, bias)



class QNetwork(nn.Module):
    def __init__(self, state_size,  action_size, seed):
        super().__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

        self.act = nn.ReLU()
        # code here

    def forward(self, state):
        return self.fc3(self.act(self.fc2(self.act(self.fc1(state)))))


class DuelingQNetwork(nn.Module):
    def __init__(self, state_size, action_size, seed):
        super().__init__()
        self.seed = torch.manual_seed(seed)

        self.feature_extractor = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU()
        )
        self.values = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.advantages = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )
        # code here

    def forward(self, state):
        y = self.feature_extractor(state)
        v = self.values(y)
        a = self.advantages(y)
        Q = v + (a - a.mean(dim=1, keepdim=True))
        return Q
    
class DuelingNoisyQNetwork(nn.Module):
    def __init__(self, state_size, action_size, seed):
        super().__init__()
        self.seed = torch.manual_seed(seed)

        self.feature_extractor = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU()
        )

        self.values_1 = NoisyLinear(64, 64)
        self.values_2 = NoisyLinear(64, 1)

        self.advantages_1 = NoisyLinear(64,64)
        self.advantages_2 = NoisyLinear(64,action_size)

        # code here
        self.act = nn.ReLU()

    def forward(self, state):
        y = self.feature_extractor(state)
        v = self.values_2(self.act(self.values_1(y)))
        a = self.advantages_2(self.act(self.advantages_1(y)))
        Q = v + (a - a.mean(dim=1, keepdim=True))
        return Q

    def reset_noise(self):
        self.values_1.reset_noise()
        self.values_2.reset_noise()
        self.advantages_1.reset_noise()
        self.advantages_2.reset_noise()

In [ ]:
import numpy as np
import random
from collections import namedtuple, deque
from model import QNetwork, DuelingQNetwork, DuelingNoisyQNetwork
from tree import SumTree
import torch
import torch.nn.functional as F
import torch.optim as optim

BUFFER_SIZE = int(1e5)
BATCH_SIZE = 64
GAMMA = 0.99
TAU = 1e-3
LR = 5e-4
UPDATE_EVERY = 4

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


class Agent():
    def __init__(self, state_size, action_size, seed, n_step=3):
        self.state_size = state_size
        self.action_size = action_size
        self.seed = random.seed(seed)
        self.n_step = n_step
        self.n_step_buffer = deque(maxlen=self.n_step)
        # Q-Network
        self.qnetwork_local = DuelingNoisyQNetwork(
            state_size, action_size, seed).to(device)
        self.qnetwork_target = DuelingNoisyQNetwork(
            state_size, action_size, seed).to(device)
        self.optimizer = optim.Adam(self.qnetwork_local.parameters(), lr=LR)
        self.memory = PrioritizingExperienceBuffer(
            self.state_size, self.action_size, BUFFER_SIZE)
        self.t_step = 0

    def _get_n_step(self):
        state, action = self.n_step_buffer[0][:2]
        reward, next_state, done = self.n_step_buffer[-1][2:]

        for transition in reversed(list(self.n_step_buffer)[:-1]):
            r, s_next, d = transition[2:]
            reward = r + GAMMA*reward*(1-d)
            if d:
                next_state, done = s_next, d

        return state, action, reward, next_state, done

    def step(self, state, action, reward, next_state, done):
        # multi-step block
        self.n_step_buffer.append((state, action, reward, next_state, done))
        if len(self.n_step_buffer) == self.n_step:
            s, a, r, s_next, d = self._get_n_step()
            self.memory.add(s, a, r, s_next, d)

        if done:
            while len(self.n_step_buffer) > 0:
                s, a, r, s_next, d = self._get_n_step()
                self.memory.add(s, a, r, s_next, d)
                self.n_step_buffer.popleft()

        # end multi step block
        self.t_step = (self.t_step + 1) % UPDATE_EVERY
        if self.t_step == 0:
            if self.memory.real_size > BATCH_SIZE:
                experiences, weights, tree_idxs = self.memory.sample(
                    BATCH_SIZE)
                self.learn(experiences, weights, tree_idxs, GAMMA)

    def act(self, state, is_eval=False):  # not using e-greedy
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        if is_eval:
            self.qnetwork_local.eval()
        else:
            self.qnetwork_local.train()
            self.qnetwork_local.reset_noise()

        with torch.no_grad():
            action_values = self.qnetwork_local(state)
        self.qnetwork_local.train()

        return np.argmax(action_values.cpu().data.numpy())

    def learn(self, experiences, weights, tree_idxs, gamma):
        '''
        experiences (Tuple[torch.Tensor]): tuple of (s, a, r, s', done) tuples
        gamma (float): discount factor
        '''
        states, actions, rewards, next_states, dones = experiences
        self.qnetwork_local.reset_noise()
        self.qnetwork_target.reset_noise()

        # classic dqn
        # q_targets_next = self.qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
        # double dqn
        q_local_best_actions = self.qnetwork_local(next_states).detach().max(
            1)[1].unsqueeze(1)  
        q_targets_next = self.qnetwork_target(next_states).detach().gather(
            1, q_local_best_actions)  

        # multi-step gamma
        gamma_n = gamma ** self.n_step
        q_target = rewards + gamma_n * q_targets_next * (1-dones)

        q_expected = self.qnetwork_local(states).gather(1, actions)

        td_error = torch.abs(q_expected - q_target).detach()

        loss = torch.mean(((q_expected-q_target)**2) * weights)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.memory.update_priorities(
            tree_idxs, td_error.cpu().numpy().squeeze())
        # TODO

        self.soft_update(self.qnetwork_local, self.qnetwork_target, TAU)

    def soft_update(self, local_model, target_model, tau):
        '''
        θ_target = τ*θ_local + (1-τ)*θ_target
        '''
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(
                tau*local_param.data + (1.0-tau)*target_param.data)


class PrioritizingExperienceBuffer:
    def __init__(self, state_size, action_size, buffer_size, eps=0.01, alpha=0.5, beta=0.1, beta_incr=1e-5):
        self.tree = SumTree(size=buffer_size)

        self.eps = eps
        self.alpha = alpha
        self.beta = beta
        self.max_priority = eps
        self.beta_incr = beta_incr

        self.state = torch.empty(buffer_size, state_size, dtype=torch.float)
        self.action = torch.empty(buffer_size, 1, dtype=torch.long)
        self.reward = torch.empty(buffer_size, dtype=torch.float)
        self.next_state = torch.empty(
            buffer_size, state_size, dtype=torch.float)
        self.done = torch.empty(buffer_size, dtype=torch.float)

        self.count = 0
        self.real_size = 0
        self.size = buffer_size

    def add(self, state, action, reward, next_state, done):
        self.tree.add(self.max_priority, self.count)

        self.state[self.count] = torch.as_tensor(state)
        self.action[self.count] = torch.as_tensor(action)
        self.reward[self.count] = torch.as_tensor(reward)
        self.next_state[self.count] = torch.as_tensor(next_state)
        self.done[self.count] = torch.as_tensor(done)

        self.count = (self.count + 1) % self.size
        self.real_size = min(self.size, self.real_size + 1)

    def sample(self, batch_size):
        assert self.real_size >= batch_size
        self.beta = min(1.0, self.beta+self.beta_incr)
        sample_idxs, tree_idxs = [], []
        priorities = torch.empty(batch_size, 1, dtype=torch.float)

        segment = self.tree.total / batch_size
        for i in range(batch_size):
            a, b = segment * i, segment * (i + 1)
            while True:
                cumsum = random.uniform(a, b)
                tree_idx, priority, sample_idx = self.tree.get(cumsum)
                if sample_idx is not None: break
            priorities[i] = float(priority)
            tree_idxs.append(tree_idx)
            sample_idxs.append(sample_idx)

        probs = priorities / self.tree.total
        weights = (self.real_size * probs) ** -self.beta
        weights = weights / weights.max()

        batch = (
            self.state[sample_idxs].to(device),
            self.action[sample_idxs].to(device),
            self.reward[sample_idxs].unsqueeze(1),
            self.next_state[sample_idxs],
            self.done[sample_idxs].unsqueeze(1)
        )
        return batch, weights, tree_idxs

    def update_priorities(self, data_idxs, priorities):
        for data_idx, priority in zip(data_idxs, priorities):
            priority = (priority + self.eps) ** self.alpha
            self.tree.update(data_idx, priority)
            self.max_priority = max(self.max_priority, priority)

class ReplayBuffer:
    def __init__(self, action_size, buffer_size, batch_size, seed):
        self.action_size = action_size
        self.memory = deque(maxlen=buffer_size)
        self.batch_size = batch_size
        self.seed = random.seed(seed)
        self.experience = namedtuple('Experience', field_names=[
                                     'state', 'action', 'reward', 'next_state', 'done'])

    def add(self, state, action, reward, next_state, done):
        e = self.experience(state, action, reward, next_state, done)
        self.memory.append(e)

    def sample(self):
        experiences = random.sample(self.memory, k=self.batch_size)
        states = torch.from_numpy(
            np.vstack([e.state for e in experiences if e is not None])).float().to(device)
        actions = torch.from_numpy(
            np.vstack([e.action for e in experiences if e is not None])).long().to(device)
        rewards = torch.from_numpy(
            np.vstack([e.reward for e in experiences if e is not None])).float().to(device)
        next_states = torch.from_numpy(np.vstack(
            [e.next_state for e in experiences if e is not None])).float().to(device)
        dones = torch.from_numpy(
            np.vstack([e.done for e in experiences if e is not None])).float().to(device)

        return (states, actions, rewards, next_states, dones)

    def __len__(self):
        return len(self.memory)


In [ ]:
class SumTree:
    def __init__(self, size):
        self.nodes = [0] * (2 * size - 1)
        self.data = [None] * size

        self.size = size
        self.count = 0
        self.real_size = 0

    @property
    def total(self):
        return self.nodes[0]

    def update(self, data_idx, value):
        idx = data_idx + self.size - 1
        change = value - self.nodes[idx]

        self.nodes[idx] = value

        parent = (idx - 1) // 2
        while parent >= 0:
            self.nodes[parent] += change
            parent = (parent - 1) // 2

    def add(self, value, data):
        self.data[self.count] = data
        self.update(self.count, value)

        self.count = (self.count + 1) % self.size
        self.real_size = min(self.size, self.real_size + 1)

    def get(self, cumsum):
        assert cumsum <= self.total

        idx = 0
        while 2 * idx + 1 < len(self.nodes):
            left, right = 2*idx + 1, 2*idx + 2

            if cumsum <= self.nodes[left]:
                idx = left
            else:
                idx = right
                cumsum = cumsum - self.nodes[left]

        data_idx = idx - self.size + 1

        return data_idx, self.nodes[idx], self.data[data_idx]

In [ ]:
%load_ext autoreload
%autoreload 2

from dqn_agent import Agent

import gymnasium as gym
from collections import deque
import numpy as np
import torch
import matplotlib.pyplot as plt
env = gym.make("LunarLander-v3")
state_size = env.observation_space.shape[0]  
action_size = env.action_space.n

agent = Agent(state_size, action_size, seed=1)

import time
def train(episodes=1500, max_t=600):
    scores=[]
    scores_window = deque(maxlen=100)

    for episode in range(1, episodes+1):
        state, info = env.reset()
        score = 0

        for t in range(max_t):
            action = agent.act(state) 
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated 
            agent.step(state, action, reward, next_state, done)

            state=next_state
            score+=reward
            if done: break
        
        scores_window.append(score)
        scores.append(score)
        


        
        if episode % 100 == 0:
            print(f'Episode: {episode}\nMean reward: {np.mean(scores_window):.2f}', end="")
        
        if episode==episodes:
            torch.save(agent.qnetwork_local.state_dict(), 'final_network.pth')
    
    return scores
        
start=time.time()
scores = train()
print(f'Train time: {time.time() - start}')
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('reward')
plt.xlabel('episode')
plt.show()

env.close()